# MERIT full-native 512: checkpointing experiment

Enable a GPU, then **Run all**. Kaggle also requires Internet. No CT data is needed.
This runtime-only experiment keeps the full 11x512x512 input, parameter shapes,
all TEP/MER/PCG modules, attention equations and loss. It adds recomputation of
activations in MER query blocks and stateless backbone regions. BatchNorm is
not recomputed, so its running statistics are updated only once.

Before the full-grid trial, this notebook compares original vs checkpointed
outputs, input/parameter gradients and normalization buffers on compact CPU/GPU
fixtures, including AMP. A failed comparison stops the full-grid trial. Passing
compact checks is evidence, not a guarantee of bitwise equality at every size.

The published model commit is unchanged. The experimental helper is embedded
below and its SHA-256 is saved with the report. The notebook writes no production
source files, changes no patient data and performs no accuracy evaluation.

In [ ]:
# Defaults are ready for Run all. No edits needed for the 512x512 experiment.
REPOSITORY = "https://github.com/torusama/merit-covid-ct.git"
SOURCE_COMMIT = "e3493dd9f2dac704d2def6225d76f19df3262ec4"
GRID_SIZE = 512
RUN_128_SANITY = False
CUDA_MEMORY_FRACTION = 1.0  # Full physical allocator budget for this experiment.

## 1. Check the assigned GPU
This short-lived child process releases its CUDA context on exit. The notebook
kernel itself does not import Torch or hold model/optimizer tensors.

In [ ]:
import json
import subprocess
import sys
from importlib import metadata
from packaging.specifiers import SpecifierSet

def runtime_probe():
    source = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': available,
    'gpus': [{'name': torch.cuda.get_device_name(i),
              'vram_mib': torch.cuda.get_device_properties(i).total_memory / 2**20}
             for i in range(torch.cuda.device_count())]}))
"""
    result = subprocess.run([sys.executable, "-c", source], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("The runtime's PyTorch could not load. Select a current GPU runtime.\n" + result.stderr)
    info = json.loads(result.stdout)
    if not info["cuda_available"]:
        raise RuntimeError("No CUDA GPU assigned. Enable a GPU in the notebook settings, reconnect, and Run all.")
    if not SpecifierSet(">=2.4.1").contains(info["torch"], prereleases=True):
        raise RuntimeError("This project needs Torch >=2.4.1. Use a newer runtime; this notebook will not replace Torch/CUDA.")
    return info

ENVIRONMENT = runtime_probe()
print(json.dumps(ENVIRONMENT, indent=2))

## 2. Fetch code and reuse installed packages
Code is cached inside the current runtime. No dataset is downloaded or inspected.
Torch and NumPy versions are constrained to the versions already supplied by the
platform; the setup cannot silently replace them. All model runs use fresh child
processes, so installing a missing library does not require restarting the kernel.

In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path
import uuid

BASE = (Path(os.environ["MERIT_CLOUD_HOME"]) if os.environ.get("MERIT_CLOUD_HOME") else
        Path("/kaggle/working") if Path("/kaggle/working").is_dir() else
        Path("/content") if Path("/content").is_dir() else Path.cwd() / "outputs" / "cloud_quickstart")
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / ("merit-source-" + SOURCE_COMMIT[:12])
RUN = BASE / "merit-results" / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])
RUN.mkdir(parents=True, exist_ok=False)

def git(*args, check=True):
    return subprocess.run(["git", "-C", str(PROJECT), *args], check=check,
                          capture_output=True, text=True)

if not PROJECT.exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPOSITORY, str(PROJECT)], check=True)
elif not (PROJECT / ".git").is_dir():
    raise RuntimeError(f"Source path exists but is not a Git checkout: {PROJECT}")
if git("status", "--porcelain", "--untracked-files=no").stdout.strip():
    raise RuntimeError("The cached source has local edits. Save those edits and select a fresh source directory.")
if git("cat-file", "-e", SOURCE_COMMIT + "^{commit}", check=False).returncode:
    git("fetch", "--quiet", "--depth", "1", "origin", SOURCE_COMMIT)
git("checkout", "--quiet", "--detach", SOURCE_COMMIT)
assert git("rev-parse", "HEAD").stdout.strip() == SOURCE_COMMIT
assert (PROJECT / "scripts/merit_preflight.py").is_file()

requirements = {
    "monai": "==1.5.2", "einops": "==0.8.2", "numpy": ">=1.24,<3",
    "nibabel": ">=5", "scipy": ">=1.10", "PyYAML": ">=6",
}
missing = []
for name, bounds in requirements.items():
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not SpecifierSet(bounds).contains(installed, prereleases=True):
        missing.append(name + bounds)

protected = {name: metadata.version(name) for name in ("torch", "numpy")}
constraints = RUN / "runtime_constraints.txt"
constraints.write_text("".join(name + "==" + version + "\n" for name, version in protected.items()))
if missing:
    print("Installing only:", ", ".join(missing), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                    "--constraint", str(constraints), *missing], check=True)
else:
    print("All project dependencies already satisfy the requirements; skipping pip.")
for name, version in protected.items():
    if metadata.version(name) != version:
        raise RuntimeError(f"Protected runtime package changed unexpectedly: {name}")
ENVIRONMENT["packages"] = {name: metadata.version(name) for name in requirements}
ENVIRONMENT["source_commit"] = SOURCE_COMMIT
ENVIRONMENT["source_repository"] = REPOSITORY
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
print("Source:", PROJECT)
print("Reports:", RUN)

## 3. Check numerical equivalence, then profile 512x512
The next source cell installs the experimental helper only inside this run's
report folder. The worker loads the unchanged pinned model and patches its
execution in memory. The notebook kernel does not hold model or CUDA tensors.

Read `profile_512_equivalence.json` for numerical checks and `profile_512.json`
for memory/optimizer results. **PASS** requires two successful optimizer steps;
**OOM** or **FAIL** is recorded and included in the final ZIP. Full training
feasibility is not established by a successful forward pass alone.

In [ ]:
TRIAL_SOURCE = r'''"""Experimental activation checkpointing; production model source is unchanged.

Run in a fresh process. The cloud notebook embeds this file so unpublished
runtime experiments can be tested against its pinned public model source.
"""

import argparse
from copy import deepcopy
import gc
import json
import os
from pathlib import Path
import sys
import types

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import torch
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

from merit_covid.models.merit_unetrpp import MERITUNETRPP, MERITLoss, MasterEvidenceRouting
from merit_covid.models.unetrpp.epa import EPA
from merit_covid.models.merit_unetrpp.modules import (
    _check_features, master_positions, relative_offsets, sinusoidal_encoding,
)


def checkpointed_mer(self, features, master_index, z_offsets_mm=None):
    """Original MER equations; recompute distance/bias/attention per query block.

    The block still attends ALL context keys in one softmax. Large distances
    must be constructed inside the checkpoint, not saved as checkpoint inputs.
    Loop-dependent coordinates are explicit inputs, not late-bound closures.
    """
    _check_features(features, self.channels)
    batch, channels, depth, height, width = features.shape
    if depth < 2:
        raise ValueError("MER needs at least one context slice")
    master = master_positions(master_index, batch, depth, features.device)
    rows = torch.arange(batch, device=features.device)
    positions = torch.arange(depth, device=features.device).expand(batch, -1)
    context_indices = positions[positions != master[:, None]].reshape(batch, depth - 1)
    slices = features.permute(0, 2, 3, 4, 1).reshape(batch, depth, height * width, channels)
    master_features = slices[rows, master]
    context = slices[rows[:, None], context_indices]
    offsets = relative_offsets(master, depth, mode=self.position_mode,
        z_offsets_mm=z_offsets_mm, unit_mm=self.position_unit_mm)[rows[:, None], context_indices]
    pe = sinusoidal_encoding(offsets, channels, dtype=context.dtype)[:, :, None]
    q = self.query(master_features)
    k = self.key(context + self.position_scale * pe).flatten(1, 2)
    v = self.value(context).flatten(1, 2)
    coords = torch.stack(torch.meshgrid(torch.arange(height, device=features.device),
        torch.arange(width, device=features.device), indexing="ij"), -1).reshape(-1, 2)

    def attention(query, keys, values, query_coords, all_coords, scale_raw):
        delta = (query_coords[:, None] - all_coords[None]).square().sum(-1)
        distance = (delta / float(height ** 2 + width ** 2)).repeat(1, depth - 1)
        bias = -F.softplus(scale_raw) * distance
        return F.scaled_dot_product_attention(query, keys, values, attn_mask=bias, dropout_p=0.0)

    updates = []
    for start in range(0, height * width, self.query_chunk_size):
        stop = start + self.query_chunk_size
        args = (q[:, start:stop], k, v, coords[start:stop], coords, self.distance_scale_raw)
        update = (checkpoint(attention, *args, use_reentrant=False, preserve_rng_state=True)
                  if self.training and torch.is_grad_enabled() else attention(*args))
        updates.append(update)
    evidence = self.output(torch.cat(updates, dim=1))
    gate = self.gate(torch.cat((master_features, evidence), -1))
    updated = master_features + self.beta * gate * evidence
    result = slices.clone()
    result[rows, master] = updated
    return result.reshape(batch, depth, height, width, channels).permute(0, 4, 1, 2, 3).contiguous()


def enable_checkpointing(model):
    """Patch execution only; keep parameters, state keys and model config intact."""
    if getattr(model, "_memory_trial_enabled", False):
        return model
    before = {name: id(value) for name, value in model.state_dict(keep_vars=True).items()}
    if model.mer is not None:
        model.mer.forward = types.MethodType(checkpointed_mer, model.mer)

    def wrap(module):
        if any(getattr(child, "track_running_stats", False) for child in module.modules()):
            raise ValueError("Checkpoint region contains mutable normalization statistics")
        original = module.forward
        def forward(*args, **kwargs):
            if not module.training or not torch.is_grad_enabled():
                return original(*args, **kwargs)
            return checkpoint(original, *args, use_reentrant=False, preserve_rng_state=True, **kwargs)
        module.forward = forward

    # Transformer conv51 contains BatchNorm even with norm_name='instance'.
    # Checkpoint only its stateless EPA, leaving BatchNorm and its updates alone.
    regions = [module for module in model.modules() if isinstance(module, EPA)]
    regions += [model.encoder1, model.decoder2]
    if model.tep is not None:
        regions.append(model.tep)
    for module in regions:
        wrap(module)
    after = {name: id(value) for name, value in model.state_dict(keep_vars=True).items()}
    if before != after:
        raise AssertionError("Checkpoint execution patch changed model state identities")
    model._memory_trial_enabled = True
    return model


def _compare_grads(reference, trial, *, rtol, atol):
    errors = []
    for (name, left), (other_name, right) in zip(reference.named_parameters(), trial.named_parameters()):
        assert name == other_name
        if left.grad is None or right.grad is None:
            raise AssertionError(f"Missing parameter gradient: {name}")
        torch.testing.assert_close(left.grad, right.grad, rtol=rtol, atol=atol, msg=name)
        if not torch.isfinite(right.grad).all():
            raise AssertionError(f"Nonfinite parameter gradient: {name}")
        errors.append(float((left.grad - right.grad).abs().max()))
    return max(errors, default=0.)


def mer_equivalence():
    torch.manual_seed(123)
    reference = MasterEvidenceRouting(4, attention_dim=8, gate_hidden=3, query_chunk_size=5).double().train()
    trial = deepcopy(reference)
    trial.forward = types.MethodType(checkpointed_mer, trial)
    x = torch.randn(2, 4, 3, 4, 6, dtype=torch.float64, requires_grad=True)
    y = x.detach().clone().requires_grad_()
    expected = reference(x, [0, 2])
    actual = trial(y, [0, 2])
    torch.testing.assert_close(expected, actual, rtol=1e-10, atol=1e-12)
    expected.square().mean().backward()
    actual.square().mean().backward()
    torch.testing.assert_close(x.grad, y.grad, rtol=1e-10, atol=1e-12)
    error = _compare_grads(reference, trial, rtol=1e-10, atol=1e-12)
    return {"status": "PASS", "dtype": "float64", "off_center_masters": True,
            "max_parameter_gradient_abs_error": error}


def model_equivalence(device, amp=False):
    torch.manual_seed(321)
    reference = MERITUNETRPP(img_size=(11, 32, 32), feature_size=2, depths=(1, 1, 1, 1)).to(device).train()
    trial = enable_checkpointing(deepcopy(reference))
    assert list(reference.state_dict()) == list(trial.state_dict())
    assert sum(p.numel() for p in reference.parameters()) == sum(p.numel() for p in trial.parameters())
    x = torch.randn(1, 1, 11, 32, 32, device=device, requires_grad=True)
    target = torch.zeros_like(x)
    target[:, :, 4:7, 5:9, 7:12] = 1
    criterion = MERITLoss()
    rng = torch.get_rng_state()
    cuda_rng = torch.cuda.get_rng_state_all() if device.type == "cuda" else None
    def evaluate(model, image):
        torch.set_rng_state(rng)
        if cuda_rng is not None:
            torch.cuda.set_rng_state_all(cuda_rng)
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
            output = model(image, [5])
            losses = criterion(output, target, [5])
        losses["total"].backward()
        return output, losses
    y = x.detach().clone().requires_grad_()
    expected, expected_loss = evaluate(reference, x)
    actual, actual_loss = evaluate(trial, y)
    rtol, atol = (1e-3, 1e-5) if amp else (1e-4, 1e-6)
    for left, right in zip(expected, actual):
        torch.testing.assert_close(left, right, rtol=rtol, atol=atol)
    for name in expected_loss:
        torch.testing.assert_close(expected_loss[name], actual_loss[name], rtol=rtol, atol=atol)
    torch.testing.assert_close(x.grad, y.grad, rtol=rtol, atol=atol)
    error = _compare_grads(reference, trial, rtol=rtol, atol=atol)
    for name, value in reference.named_buffers():
        torch.testing.assert_close(value, dict(trial.named_buffers())[name], rtol=0, atol=0, msg=name)
    return {"status": "PASS", "amp": amp, "device": str(device), "rtol": rtol, "atol": atol,
            "same_parameter_count_and_state_keys": True, "dropout_rng_preserved": True,
            "normalization_buffers_unchanged": True,
            "max_output_abs_error": max(float((a - b).abs().max().detach()) for a, b in zip(expected, actual)),
            "max_input_gradient_abs_error": float((x.grad - y.grad).abs().max()),
            "max_parameter_gradient_abs_error": error}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--size", type=int, default=512)
    parser.add_argument("--memory-fraction", type=float, default=1.)
    parser.add_argument("--validate-only", action="store_true")
    args = parser.parse_args()
    if not 0 < args.memory_fraction <= 1:
        parser.error("memory fraction must be in (0,1]")
    args.output.parent.mkdir(parents=True, exist_ok=True)
    if args.output.exists():
        raise FileExistsError(args.output)
    torch.set_num_threads(2)
    torch.use_deterministic_algorithms(True)
    device = torch.device("cuda:0")
    torch.cuda.set_per_process_memory_fraction(args.memory_fraction, 0)
    validation = {"status": "FAIL"}
    path = args.output.with_name(args.output.stem + "_equivalence.json")
    if path.exists():
        raise FileExistsError(path)
    try:
        validation["mer"] = mer_equivalence()
        validation["model_fp32"] = model_equivalence(device)
        validation["model_amp"] = model_equivalence(device, amp=True)
        validation["status"] = "PASS"
    except Exception as error:
        validation["error"] = str(error)
        raise
    finally:
        path.write_text(json.dumps(validation, indent=2) + "\n", encoding="utf-8")
    print("Output/gradient equivalence: PASS", flush=True)
    if args.validate_only:
        return 0
    gc.collect()
    torch.cuda.empty_cache()
    from merit_covid.engine import preflight
    original_class = preflight.MERITUNETRPP
    def factory(**kwargs):
        return enable_checkpointing(original_class(**kwargs))
    preflight.MERITUNETRPP = factory
    report = preflight.profile_model(args.config, args.output, size=args.size, batch_size=1,
        device="cuda", warmup=1, repeats=1)
    report.update(checkpoint_trial=True, equivalence=validation,
        cloud_allocator_fraction=args.memory_fraction, production_model_source_modified=False,
        execution_regions="MER query blocks including bias; EPA, TEP, encoder1 and final decoder; BatchNorm excluded")
    args.output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    print("Checkpoint trial:", report["status"], flush=True)
    return 0 if report["status"] == "PASS" else 1


if __name__ == "__main__":
    raise SystemExit(main())
'''

import hashlib
TRIAL_FILE = RUN / "merit_checkpoint_trial.py"
TRIAL_FILE.write_text(TRIAL_SOURCE, encoding="utf-8")
ENVIRONMENT["checkpoint_trial_sha256"] = hashlib.sha256(TRIAL_SOURCE.encode("utf-8")).hexdigest()
ENVIRONMENT["checkpoint_trial"] = True
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
print("Experimental helper SHA-256:", ENVIRONMENT["checkpoint_trial_sha256"])


In [ ]:
if type(GRID_SIZE) is not int or GRID_SIZE < 32 or GRID_SIZE % 32:
    raise ValueError("GRID_SIZE must be a positive multiple of 32.")
if not 0 < CUDA_MEMORY_FRACTION <= 1:
    raise ValueError("CUDA_MEMORY_FRACTION must be in (0,1].")

worker = TRIAL_FILE
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
results = []
sizes = ([128] if RUN_128_SANITY and GRID_SIZE != 128 else []) + [GRID_SIZE]
for size in sizes:
    destination = RUN / f"profile_{size}.json"
    command = [sys.executable, str(worker), "--config", str(PROJECT / "configs/merit_index.yaml"),
               "--output", str(destination), "--size", str(size),
               "--memory-fraction", str(CUDA_MEMORY_FRACTION)]
    print(f"Starting {size}x{size}, batch 1, AMP, complete synthetic training steps...", flush=True)
    with (RUN / f"profile_{size}.log").open("w") as log:
        process = subprocess.Popen(command, cwd=PROJECT, env=environment, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        exit_code = process.wait()
    if destination.exists():
        report = json.loads(destination.read_text())
    else:
        report = {"status": "FAIL", "error": "Worker produced no report; inspect its log.", "optimizer_steps": 0}
    row = {"grid": size, "status": report["status"], "exit_code": exit_code,
           "optimizer_steps": report.get("optimizer_steps", 0),
           "peak_allocated_mib": report.get("peak_allocated_mib"),
           "peak_reserved_mib": report.get("peak_reserved_mib"),
           "mean_step_seconds": report.get("mean_step_seconds"),
           "parameter_count": report.get("parameter_count"), "error": report.get("error")}
    results.append(row)
    (RUN / "summary.json").write_text(json.dumps(results, indent=2) + "\n")
    print(json.dumps(row, indent=2), flush=True)
    if size != GRID_SIZE and report["status"] != "PASS":
        print("Sanity check failed; the 512-grid attempt was not started. Read the saved report.")
        break

## 4. Save the report
The ZIP contains GPU/software information, source commit, effective configs,
logs, successful optimizer-step counts, timing and memory/error reports. Kaggle
stores it under `/kaggle/working`; use **Save Version** to retain outputs. On
Colab, run the download cell below before the runtime is discarded. If the
browser does not save the file, open the left Files panel, refresh, expand
`merit-results`, then use the ZIP file's three-dot menu > Download. The printed
`/content/...` path is a runtime filesystem path, not a browser download URL.
No patient data, trained model or credentials are included.

If 512 passes, the next step is a bounded real-CT pilot before full training.
Do not interpret synthetic throughput or loss as segmentation quality.

In [ ]:
import shutil
archive = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
print("Report ZIP:", archive)
print("Download using the next cell, or the Files panel > ZIP menu > Download.")

In [ ]:
# On Colab this downloads the completed report. On Kaggle use the Output files.
try:
    import google.colab
except ImportError:
    print("Open the report ZIP from the notebook's output files:", archive)
else:
    from google.colab import files
    files.download(str(archive))
    print("If no download appears, open Files on the left, refresh, then merit-results > ZIP menu > Download.")